# 7주차 실습 — 개체명 인식, N-gram 언어 모델, 정보 추출

**자연어처리 | Chapter 7 실습 노트북**

---

### 이번 실습의 목표

문장에서 유용한 정보를 어떻게 뽑아내는지, 세 가지 과제를 **직접 코드로 실행**해 보고 각 단계의 출력이 다음 단계의 입력이 되는 과정을 눈으로 확인한다.

| PART | 주제 | 대응 절 | 주요 도구 |
|:---:|---|:---:|---|
| 1 | 개체명 인식 (NER) | 7.1 | 사전·규칙 기반 NER, BIO 태깅, 개체 단위 P/R/F1 |
| 2 | N-gram 언어 모델 | 7.2 | MLE, 라플라스 스무딩, 로그확률, 문장 생성, 퍼플렉서티 |
| 3 | 정보 추출 (IE) | 7.3 | 패턴 기반 관계 추출, 지식 그래프, Hearst 패턴 |

### 진행 방법

1. 아래 **[0] 환경 설정** 셀을 먼저 실행한다.
2. 위에서부터 순서대로 셀을 실행한다.
3. 노트북 뒷부분의 **[과제] 3문항** 이 제출 대상이다.
4. **실행 결과(출력)가 모두 포함된 상태**로 저장하여 제출한다. 파일명 : `7주차_실습_학번_이름.ipynb`

> **실행 환경** : Google Colab 권장. 표준 라이브러리만 사용하므로 로컬 주피터에서도 동일하게 동작한다.

> **[문제] 노트북 사용법**
>
> - `raise NotImplementedError(...)` 로 막혀 있는 부분이 여러분이 채워야 할 곳이다.
>   그 줄을 지우고 직접 코드를 작성한다.
> - 아직 구현하지 않은 과제가 있어도 `런타임 → 모두 실행` 은 끝까지 동작한다.
>   미구현 과제가 있는 셀은 `▶ 미구현 : ...` 이라고만 출력된다.
> - 각 과제 아래의 **채점 셀** 이 통과(`통과!` 출력)해야 정답으로 인정된다.

## [0] 환경 설정

이번 실습에서 사용하는 라이브러리는 다음과 같다.

- **파이썬 표준 라이브러리** (`re`, `math`, `random`, `collections`) — NER 규칙, n-gram 계수, 관계 추출 패턴을 모두 표준 라이브러리만으로 구현한다. 별도 설치가 필요 없다.
- **matplotlib** — 결과를 그림으로 확인할 때 사용한다. 한글 라벨이 깨지지 않도록 나눔 폰트를 설치하고 등록한다. &nbsp;→ *Colab에 이미 설치되어 있다*
- **NLTK** — 영어 토큰화(`punkt`)에만 **선택적으로** 쓴다. 설치되어 있지 않거나 데이터를 내려받지 못해도 아래 `[0-3]` 의 **정규식 토크나이저로 자동 대체**되므로 실습 전체가 정상 진행된다. &nbsp;→ *따로 설치하지 않는다*

> **중요** : 이미 설치된 패키지를 `pip` 로 다시 설치하면 안 된다.
> 의존 패키지가 교체되면서 "런타임을 다시 시작하라"는 안내가 뜨고, 재시작하면 앞에서 실행한 셀의 변수가 모두 사라진다.
> 그래서 아래 셀에서는 **Colab에 없는 한글 폰트만** 설치한다.

아래 셀은 한 번만 실행하면 된다.

In [ ]:
# ── [0-1] 라이브러리 설치 (Colab 기준) ────────────────────────────
# 이 노트북은 파이썬 표준 라이브러리만으로 동작하므로 새로 설치할 파이썬 패키지가 없다.
# (NLTK 는 선택 사항이며, 없으면 [0-3] 의 정규식 토크나이저로 자동 대체된다)

# 그래프의 한글이 깨지지 않도록 나눔 폰트만 설치한다.
!apt-get -qq install -y fonts-nanum > /dev/null 2>&1

print("설치 완료")

In [ ]:
# ── [0-2] 공통 import 및 한글 폰트 설정 ───────────────────────────
import sys, re, math, random, collections
from collections import Counter, defaultdict

random.seed(42)          # 문장 생성 결과를 재현 가능하게 고정한다
print("Python:", sys.version.split()[0])

# 한글 폰트 설정 ---------------------------------------------------------
# 주의 : apt 로 폰트를 설치해도 matplotlib 의 폰트 캐시는 갱신되지 않기 때문에
#        plt.rcParams['font.family'] 만 바꾸면 그래프의 한글이 그대로 깨진다.
#        addfont() 로 폰트 파일을 직접 등록해야 재시작 없이 즉시 적용된다.
import os
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

NANUM = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
if os.path.exists(NANUM):
    fm.fontManager.addfont(NANUM)
    plt.rcParams['font.family'] = 'NanumGothic'
else:
    for cand in ['Malgun Gothic', 'AppleGothic', 'NanumGothic', 'Noto Sans CJK KR']:
        if any(f.name == cand for f in fm.fontManager.ttflist):
            plt.rcParams['font.family'] = cand
            break
plt.rcParams['axes.unicode_minus'] = False

print('matplotlib 한글 폰트 :', plt.rcParams['font.family'])

# NLTK 준비 (선택 사항) --------------------------------------------------
# NLTK 가 없거나 punkt 데이터를 내려받지 못해도 실습은 그대로 진행된다.
# 그 경우 [0-3] 의 tokenize_en() 이 정규식 토크나이저로 자동 대체된다.
NLTK_OK = False
try:
    import nltk
    NLTK_OK = True
except ImportError:
    print("NLTK 를 불러올 수 없다 (정규식 토크나이저로 대체).")

PUNKT_OK = False
if NLTK_OK:
    try:
        nltk.download("punkt", quiet=True)
        nltk.download("punkt_tab", quiet=True)
        from nltk.tokenize import word_tokenize
        word_tokenize("This is a test.")
        PUNKT_OK = True
    except Exception as e:
        print("NLTK 데이터 내려받기 실패 (정규식 토크나이저로 대체):", e)

print("NLTK 사용 가능:", NLTK_OK, "/ punkt 사용 가능:", PUNKT_OK)

In [ ]:
# 공통 토크나이저 : NLTK가 있으면 NLTK, 없으면 정규식 기반으로 동작
_TOKEN_RE = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?|[0-9]+(?:[.,][0-9]+)?|[가-힣]+|[^\s]")

def tokenize_en(text):
    """영어 문장을 토큰 리스트로 변환한다."""
    if PUNKT_OK:
        from nltk.tokenize import word_tokenize
        return word_tokenize(text)
    return _TOKEN_RE.findall(text)

def tokenize_ko(text):
    """한국어 문장을 어절(띄어쓰기) 단위로 나누되, 문장부호는 분리한다."""
    text = re.sub(r"([,.!?])", r" \1 ", text)
    return [t for t in text.split() if t]

print(tokenize_en("Jim bought 300 shares of Acme Corp. in 2006."))
print(tokenize_ko("춘향아, 9월 15일에 강남에서 홍길동과 약속이 있으니까!"))

---
# 실습 1. 개체명 인식 (Named Entity Recognition)

강의에서 다룬 내용
- 개체명 : 사람(PS/PER), 장소(LC/LOC), 기관(OG/ORG), 날짜(DT/DATE) 등 고유한 의미를 갖는 개체
- BIO 태깅 : B(개체 시작), I(개체 내부), O(개체 아님)
- 평가 : 개체 단위 Precision / Recall / F1

### 1-1. 실습용 데이터

작은 한국어 코퍼스를 사용한다. 각 문장은 `(토큰 리스트, 정답 BIO 태그 리스트)` 형태이다.

In [ ]:
# (토큰, 정답 BIO 태그) 형태의 소형 한국어 NER 데이터셋
GOLD_DATA = [
    (["춘향아", ",", "9월", "15일에", "강남에서", "홍길동과", "약속이", "있으니까", "!"],
     ["B-PS",  "O", "B-DT", "I-DT",   "B-LC",    "B-PS",    "O",      "O",       "O"]),

    (["명량", "대첩은", "1597년", "정유재란", "때", "이순신이", "지휘한", "해전이다", "."],
     ["B-EV", "I-EV", "B-DT",  "B-EV",   "O",  "B-PS",   "O",     "O",      "O"]),   # 강의자료 9쪽과 동일하게 '명량 대첩'을 사건(EV)으로 태깅

    (["홍길동은", "2020년부터", "한국대학교", "컴퓨터공학과", "교수로", "재직", "중이다", "."],
     ["B-PS",   "B-DT",     "B-OG",     "I-OG",       "O",    "O",   "O",    "O"]),

    (["삼성전자는", "수원에", "본사를", "두고", "있다", "."],
     ["B-OG",     "B-LC",  "O",    "O",   "O",   "O"]),

    (["이순신", "장군은", "1545년", "서울에서", "태어났다", "."],
     ["B-PS",  "O",    "B-DT",  "B-LC",   "O",       "O"]),

    (["춘향과", "몽룡은", "남원", "광한루에서", "만났다", "."],
     ["B-PS",  "B-PS", "B-LC", "I-LC",     "O",     "O"]),
]

print("문장 수 :", len(GOLD_DATA))
for tok, tag in GOLD_DATA[:2]:
    print()
    for t, g in zip(tok, tag):
        print(f"  {t:<10} {g}")

### 1-2. BIO 태그 <-> 개체(span) 상호 변환

평가와 결과 확인을 위해 두 표현을 자유롭게 오갈 수 있어야 한다.

- `bio_to_spans` : BIO 태그열 -> `(시작index, 끝index, 유형)` 집합
- `spans_to_bio` : span 목록 -> BIO 태그열

In [ ]:
def bio_to_spans(tags):
    """BIO 태그 리스트를 (start, end, type) 튜플 리스트로 변환한다. end는 포함(inclusive)."""
    spans, start, ent = [], None, None
    for i, tag in enumerate(tags):
        if tag.startswith("B-"):
            if start is not None:
                spans.append((start, i - 1, ent))
            start, ent = i, tag[2:]
        elif tag.startswith("I-"):
            # 앞에 같은 유형의 개체가 없으면 B로 간주 (불법 태그열 방어)
            if start is None or ent != tag[2:]:
                if start is not None:
                    spans.append((start, i - 1, ent))
                start, ent = i, tag[2:]
        else:  # "O"
            if start is not None:
                spans.append((start, i - 1, ent))
            start, ent = None, None
    if start is not None:
        spans.append((start, len(tags) - 1, ent))
    return spans


def spans_to_bio(spans, length):
    """(start, end, type) 목록을 길이 length의 BIO 태그 리스트로 변환한다."""
    tags = ["O"] * length
    for s, e, t in spans:
        tags[s] = "B-" + t
        for i in range(s + 1, e + 1):
            tags[i] = "I-" + t
    return tags


toks, tags = GOLD_DATA[2]
spans = bio_to_spans(tags)
print("토큰 :", toks)
print("태그 :", tags)
print("스팬 :", spans)
print("복원 :", spans_to_bio(spans, len(toks)))
print("왕복 일치 여부 :", spans_to_bio(spans, len(toks)) == tags)

### 1-3. 사전·규칙 기반 NER 시스템

강의에서 살펴본 **지식 기반 시스템**을 아주 작게 구현해 본다.

1. 사전(gazetteer) 일치 : 인명·지명·기관명 목록과 대조
2. 정규표현식 규칙 : 날짜 표현(`1597년`, `9월`, `15일`)
3. 접미사 규칙 : `대학교`, `전자`, `주식회사` 등으로 끝나면 기관

한국어는 교착어이므로 **조사가 붙은 어절**을 처리하기 위해, 어절의 앞부분이 사전 항목으로 시작하는지 확인한다.

In [ ]:
GAZETTEER = {
    "PS": ["춘향", "홍길동", "이순신", "몽룡", "세종"],
    "LC": ["강남", "명량", "서울", "수원", "남원", "광한루", "부산"],
    "OG": ["한국대학교", "삼성전자", "국립국어원"],
    # 주의 : '명량'은 지명(LC)이자 사건명 '명량 대첩'의 일부라 중의적이다.
    #        GAZETTEER 는 딕셔너리 순서(PS→LC→OG→EV)로 검사하므로 '명량'은 LC 로 판정된다.
    #        이 불일치는 1-5 오류 분석에서 FN/FP 사례로 직접 확인한다.
    "EV": ["정유재란", "임진왜란", "대첩"],
}

DATE_RE = re.compile(r"^[0-9]{1,4}(년|월|일)")
ORG_SUFFIX = ("대학교", "전자", "주식회사", "연구원", "공학과", "학과")

def rule_based_ner(tokens):
    """어절 리스트를 입력받아 BIO 태그 리스트를 반환하는 규칙 기반 NER."""
    tags = ["O"] * len(tokens)
    for i, tok in enumerate(tokens):
        matched = None
        # (1) 사전 일치 : 조사가 붙어 있어도 앞부분이 사전 항목이면 인정
        for ent_type, words in GAZETTEER.items():
            for w in words:
                if tok.startswith(w):
                    matched = ent_type
                    break
            if matched:
                break
        # (2) 날짜 정규표현식
        if matched is None and DATE_RE.match(tok):
            matched = "DT"
        # (3) 기관 접미사 규칙 (조사 제거 후 검사)
        if matched is None:
            # 주의 : 조사 목록에 '와/과'를 넣으면 "컴퓨터공학과" 의 '과'까지 잘려 나가
            #        ORG 접미사 규칙(공학과/학과)이 절대 발동하지 않는다. 그래서 제외한다.
            stem = re.sub(r"(은|는|이|가|을|를|의|에서|에|로|으로)$", "", tok)
            if tok.endswith(ORG_SUFFIX) or stem.endswith(ORG_SUFFIX):
                matched = "OG"
        if matched:
            # 앞 토큰이 같은 유형이면 I-, 아니면 B-
            if i > 0 and tags[i - 1] in ("B-" + matched, "I-" + matched):
                tags[i] = "I-" + matched
            else:
                tags[i] = "B-" + matched
    return tags


for toks, gold in GOLD_DATA[:3]:
    pred = rule_based_ner(toks)
    print(" ".join(toks))
    print("   정답 :", gold)
    print("   예측 :", pred)
    print()

### 1-4. 개체 단위 평가 : Precision / Recall / F1

강의에서 설명한 대로 **토큰 정확도(accuracy)는 쓰지 않는다.**  
대부분의 토큰이 `O`이므로, 전부 `O`로 예측해도 정확도가 매우 높게 나오기 때문이다.

- Precision = TP / (TP + FP) = 맞춘 개체 수 / 예측한 개체 수
- Recall = TP / (TP + FN) = 맞춘 개체 수 / 정답 개체 수
- F1 = 2PR / (P + R)

**strict(엄격) 기준** : 시작·끝·유형이 모두 일치해야 정답  
**relaxed(완화) 기준** : 유형이 같고 구간이 한 토큰이라도 겹치면 정답

In [ ]:
def evaluate_ner(data, tagger, mode="strict", verbose=True):
    """data: [(tokens, gold_tags)], tagger: tokens -> pred_tags"""
    tp = fp = fn = 0
    for tokens, gold_tags in data:
        gold = bio_to_spans(gold_tags)
        pred = bio_to_spans(tagger(tokens))
        if mode == "strict":
            g, p = set(gold), set(pred)
            hit = g & p
            tp += len(hit); fp += len(p - g); fn += len(g - p)
        else:  # relaxed : 유형이 같고 구간이 겹치면 정답
            used = set()
            hit = 0
            for ps, pe, pt in pred:
                found = False
                for k, (gs, ge, gt) in enumerate(gold):
                    if k in used or gt != pt:
                        continue
                    if not (pe < gs or ps > ge):   # 구간이 겹침
                        used.add(k); found = True; break
                hit += 1 if found else 0
            tp += hit; fp += len(pred) - hit; fn += len(gold) - hit

    prec = tp / (tp + fp) if tp + fp else 0.0
    rec  = tp / (tp + fn) if tp + fn else 0.0
    f1   = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    if verbose:
        print(f"[{mode:<7}] TP={tp:2d}  FP={fp:2d}  FN={fn:2d}  "
              f"Precision={prec:.3f}  Recall={rec:.3f}  F1={f1:.3f}")
    return prec, rec, f1


print("규칙 기반 NER 성능")
evaluate_ner(GOLD_DATA, rule_based_ner, mode="strict")
evaluate_ner(GOLD_DATA, rule_based_ner, mode="relaxed")

print()
print("비교) 모든 토큰을 O로 예측하는 시스템")
evaluate_ner(GOLD_DATA, lambda toks: ["O"] * len(toks), mode="strict")

# 토큰 정확도는 왜 부적절한가?
tot = hit = 0
for toks, gold in GOLD_DATA:
    for g in gold:
        tot += 1
        hit += 1 if g == "O" else 0
print(f"\n전부 O로 예측했을 때의 토큰 정확도 = {hit/tot:.3f}   (F1은 0.000)")
print("  * 이 실습 데이터는 개체 밀도가 매우 높아 정확도가 0.5 수준이지만,")
print("    실제 뉴스 코퍼스는 토큰의 85~90%가 O이므로 '전부 O' 시스템의 정확도가 0.9에 가깝다.")
print("    즉 정확도는 NER 성능을 전혀 반영하지 못하므로 개체 단위 F1을 사용해야 한다.")

### 1-5. 오류 분석

어떤 개체를 놓쳤고(FN), 어떤 개체를 잘못 예측했는지(FP) 직접 확인한다.  
사전에 없는 개체명은 재현율을 떨어뜨리고, 지나치게 일반적인 규칙은 정밀도를 떨어뜨린다.

In [ ]:
# 규칙 기반 NER이 어떤 개체를 놓치고(FN) 어떤 개체를 잘못 잡는지(FP) 문장 단위로 확인한다.
print("=== 오류 분석 (strict 기준) ===")
for tokens, gold_tags in GOLD_DATA:
    # 정답과 예측을 각각 (시작, 끝, 유형) 스팬 집합으로 바꾸면 집합 연산만으로 오류를 가려낼 수 있다.
    gold = set(bio_to_spans(gold_tags))
    pred = set(bio_to_spans(rule_based_ner(tokens)))
    # 놓친 개체(FN) = 정답에만 있는 스팬, 잘못 잡은 개체(FP) = 예측에만 있는 스팬
    miss, spur = gold - pred, pred - gold
    if not miss and not spur:
        continue   # 완벽히 맞춘 문장은 분석할 것이 없으므로 건너뛴다
    print("\n문장 :", " ".join(tokens))
    # FN 원인 : 다중 어절 개체를 어절 단위 사전으로 잡지 못하고('명량 대첩'),
    #           인접한 동일 유형 개체가 한 스팬으로 병합된다('춘향과 몽룡은'). → 재현율 하락
    for s, e, t in sorted(miss):
        print(f"   [놓침 FN] {' '.join(tokens[s:e+1])!r} ({t})")
    # FP 원인 : 사전 유형 충돌('명량'→LC)과 지나치게 일반적인 항목('대첩'). → 정밀도 하락
    for s, e, t in sorted(spur):
        print(f"   [오탐 FP] {' '.join(tokens[s:e+1])!r} ({t})")

---
# 실습 2. N-gram 언어 모델

강의에서 다룬 내용
- 연쇄 법칙 : `P(w1..wn) = P(w1)P(w2|w1)...P(wn|w1..wn-1)`
- 마르코프 가정 : `P(wn|w1..wn-1) ≈ P(wn|wn-N+1..wn-1)`
- MLE 추정 : `count(문맥+단어) / count(문맥)`
- 희소성 문제와 라플라스 스무딩
- 로그 확률과 퍼플렉서티

### 2-1. 학습 코퍼스 준비

강의 예제에 등장한 3문장 미니 코퍼스와, 조금 더 큰 영어 코퍼스를 함께 사용한다.  
문장 경계 기호 `<s>`, `</s>` 를 붙이는 것이 핵심이다.

In [ ]:
# n-gram 언어 모델 학습에 쓸 코퍼스를 정의하고, 문장 경계 기호를 붙이는 전처리를 확인한다.

# 강의 슬라이드의 확률 계산을 그대로 재현하기 위한 3문장 미니 코퍼스
MINI_CORPUS = [
    "I am Sam",
    "Sam I am",
    "I do not like green eggs and ham",
]

# 어휘를 서로 공유하는 10문장 코퍼스. n 값에 따라 생성 문장이 어떻게 달라지는지 관찰하는 데 쓴다.
BIG_CORPUS = [
    "the boy is looking at a pretty girl",
    "the girl is looking at a small dog",
    "a boy and a girl are walking in the park",
    "the dog is running in the park",
    "the boy is running with the dog",
    "a pretty girl is walking in the park",
    "the small dog is looking at the boy",
    "the boy and the dog are in the park",
    "a girl is reading a book in the park",
    "the boy is reading a book",
]

# 문장 시작 기호, 문장 끝 기호, 미등록어 기호
BOS, EOS, UNK = "<s>", "</s>", "<unk>"

def prepare(sentences, n):
    """문장 리스트를 토큰 리스트로 바꾸고 경계 기호를 붙인다. n-gram의 n에 맞춰 <s>를 n-1개 붙임."""
    out = []
    for s in sentences:
        # 대소문자 차이 때문에 같은 단어가 서로 다른 토큰으로 세어지지 않도록 소문자로 통일한다.
        toks = tokenize_en(s.lower())
        # 첫 단어도 (n-1)개짜리 문맥을 가질 수 있도록 <s>를 n-1개 붙이고,
        # 문장이 끝날 확률 P(</s> | ...)까지 모델링하기 위해 </s>를 하나 붙인다.
        out.append([BOS] * (n - 1) + toks + [EOS])
    return out

# 2-gram이면 <s>가 1개, 3-gram이면 2개 붙는 것을 확인한다.
print(prepare(MINI_CORPUS, 2)[0])
print(prepare(MINI_CORPUS, 3)[0])

### 2-2. N-gram 언어 모델 클래스

`NgramLM` 은 다음 기능을 제공한다.

| 메서드 | 설명 |
|---|---|
| `fit` | 코퍼스에서 n-gram과 (n-1)-gram 빈도를 센다 |
| `prob` | 조건부 확률 `P(w | context)` 을 반환 (MLE 또는 add-k) |
| `logprob_sentence` | 문장의 로그 확률 (언더플로 방지) |
| `perplexity` | 테스트 코퍼스의 퍼플렉서티 |
| `generate` | 학습된 분포에서 문장을 생성 |

In [ ]:
class NgramLM:
    def __init__(self, n=2, k=0.0):
        """n : n-gram 차수, k : add-k 스무딩 상수 (0이면 MLE)"""
        self.n = n
        self.k = k
        self.ngram = Counter()     # (w_{i-n+1},...,w_i) 빈도
        self.context = Counter()   # (w_{i-n+1},...,w_{i-1}) 빈도
        self.vocab = set()

    def fit(self, sentences):
        data = prepare(sentences, self.n)
        for toks in data:
            self.vocab.update(t for t in toks if t != BOS)
            for i in range(self.n - 1, len(toks)):
                ctx = tuple(toks[i - self.n + 1:i])
                w = toks[i]
                self.ngram[ctx + (w,)] += 1
                self.context[ctx] += 1
        self.V = len(self.vocab)
        return self

    def prob(self, word, context=()):
        ctx = tuple(context)[-(self.n - 1):] if self.n > 1 else ()
        num = self.ngram[ctx + (word,)] + self.k
        den = self.context[ctx] + self.k * self.V
        if den == 0:
            return 0.0
        return num / den

    def logprob_sentence(self, sentence):
        toks = prepare([sentence], self.n)[0]
        total, detail = 0.0, []
        for i in range(self.n - 1, len(toks)):
            ctx, w = tuple(toks[i - self.n + 1:i]), toks[i]
            p = self.prob(w, ctx)
            detail.append((ctx, w, p))
            if p <= 0:
                return float("-inf"), detail
            total += math.log(p)
        return total, detail

    def perplexity(self, sentences):
        logsum, ntok = 0.0, 0
        for s in sentences:
            lp, _ = self.logprob_sentence(s)
            if lp == float("-inf"):
                return float("inf")
            toks = prepare([s], self.n)[0]
            logsum += lp
            ntok += len(toks) - (self.n - 1)
        return math.exp(-logsum / ntok)

    def generate(self, max_len=20, seed=None):
        rng = random.Random(seed)
        ctx = tuple([BOS] * (self.n - 1))
        out = []
        for _ in range(max_len):
            if self.n == 1:
                cands = [(g[-1], c) for g, c in self.ngram.items() if g[-1] != BOS]
            else:
                cands = [(g[-1], c) for g, c in self.ngram.items() if g[:-1] == ctx]
            if not cands:
                break
            words, weights = zip(*cands)
            w = rng.choices(words, weights=weights, k=1)[0]
            if w == EOS:
                break
            out.append(w)
            ctx = (ctx + (w,))[-(self.n - 1):] if self.n > 1 else ()
        return " ".join(out)

print("NgramLM 클래스 정의 완료")

### 2-3. 바이그램 확률 직접 확인

강의 슬라이드의 계산을 코드로 재현한다.

```
P(I | <s>)   = 2/3 = 0.67
P(am | I)    = 2/3 = 0.67
P(Sam | am)  = 1/2 = 0.50
P(</s>| Sam) = 1/2 = 0.50
```

In [ ]:
# 강의 슬라이드의 바이그램 확률 계산을 코드로 재현한다.
# 핵심은 P(w | context) 가 결국 C(context, w) / C(context) 라는 빈도비라는 점이다.
bi = NgramLM(n=2).fit(MINI_CORPUS)

# 슬라이드에 제시된 (단어, 직전 문맥) 쌍 네 개
checks = [("i", ("<s>",)), ("am", ("i",)), ("sam", ("am",)), ("</s>", ("sam",))]
for w, ctx in checks:
    # 확률과 함께 분자(n-gram 빈도)와 분모(문맥 빈도)를 같이 출력해 계산 근거를 드러낸다.
    print(f"P({w:<5} | {ctx[0]:<4}) = {bi.prob(w, ctx):.4f}"
          f"   [count={bi.ngram[ctx + (w,)]}, context count={bi.context[ctx]}]")

# 문장 확률은 시점별 조건부 확률의 곱이지만, 곱하면 값이 너무 작아져 언더플로가 난다.
# 그래서 로그 확률의 합으로 계산한 뒤 마지막에 exp 를 취해 원래 확률로 되돌린다.
lp, detail = bi.logprob_sentence("I am Sam")
print("\n문장 'I am Sam' 의 계산 과정")
for ctx, w, p in detail:
    print(f"   P({w:<5} | {' '.join(ctx):<5}) = {p:.4f}")
# 네 확률 (2/3) x (2/3) x (1/2) x (1/2) = 1/9 ≈ 0.111 과 일치하는지 확인해 보자.
# (문장이 끝날 확률 P(</s> | sam) 까지 포함해야 한다는 점에 주의)
print(f"   로그 확률 = {lp:.4f}   ->   확률 = {math.exp(lp):.6f}")

### 2-4. 희소성 문제 확인

학습 코퍼스에 없는 단어 조합이 하나라도 있으면 문장 전체 확률이 **0**이 된다.  
로그 공간에서는 `-inf` 로 나타난다.

In [ ]:
test_sents = ["I am Sam", "Sam I like ham", "I like green eggs"]

print("=== MLE (스무딩 없음) ===")
for s in test_sents:
    lp, _ = bi.logprob_sentence(s)
    print(f"  {s:<20} logP = {lp}")

bi_lap = NgramLM(n=2, k=1.0).fit(MINI_CORPUS)   # 라플라스(add-1) 스무딩
print("\n=== 라플라스 스무딩 (add-1) ===")
for s in test_sents:
    lp, _ = bi_lap.logprob_sentence(s)
    print(f"  {s:<20} logP = {lp:.4f}  (P = {math.exp(lp):.3e})")

print(f"\n어휘 크기 V = {bi_lap.V}")
print("한 번도 보지 못한 조합 P(ham | like) =", round(bi_lap.prob("ham", ("like",)), 5))

### 2-5. N 값에 따른 문장 생성 비교

강의에서 본 셰익스피어 예시와 같은 현상을 작은 코퍼스에서 재현한다.

- N이 작으면 : 새롭지만 부자연스러운 문장
- N이 크면 : 자연스럽지만 학습 코퍼스를 거의 그대로 복사한 문장

In [ ]:
# n 값을 바꿔 가며 문장을 생성해 본다.
# n이 작으면 새롭지만 어색한 문장이, n이 크면 자연스럽지만 학습 코퍼스를 거의 베낀 문장이 나온다.
for n in [1, 2, 3]:
    lm = NgramLM(n=n).fit(BIG_CORPUS)
    print(f"--- {n}-gram 모델로 생성한 문장 ---")
    for seed in range(4):
        # 같은 모델에서 시드만 바꿔 서로 다른 표본 4개를 뽑는다. (시드를 고정해 결과를 재현 가능하게 한다)
        print("   ", lm.generate(max_len=15, seed=seed))
    print()

### 2-6. 퍼플렉서티(Perplexity) 비교

`PPL = exp( -(1/N) * Σ log P(wi | context) )`

- 값이 **낮을수록** 좋은 모델
- 스무딩이 없으면 미관측 조합 때문에 `inf` 가 나올 수 있다
- 학습 데이터에 대한 PPL과 테스트 데이터에 대한 PPL의 차이가 곧 **일반화 능력**이다

In [ ]:
# 학습 코퍼스에 대한 PPL 과, 학습에 없는 문장에 대한 PPL 을 비교해 일반화 능력을 확인한다.
train = BIG_CORPUS

# 테스트 문장 : 등장하는 단어는 모두 학습 코퍼스에 있지만,
#               아래 주석에 적은 bigram 은 학습 코퍼스에서 한 번도 관측되지 않았다.
test = [
    "a pretty dog is running with a small boy",   # 미관측 : (pretty,dog) (with,a) (small,boy)
    "the girl and the boy are reading a book",    # 미관측 : (girl,and) (boy,are) (are,reading)
    "a small girl is looking at the park",        # 미관측 : (small,girl)
]

# '학습에 없는 조합'이라는 말을 눈짐작으로 넘기지 말고 코드로 직접 세어 확인한다.
_bi_check = NgramLM(n=2).fit(train)
print("=== 테스트 문장의 미관측 bigram 확인 ===")
for s in test:
    toks = prepare([s], 2)[0]
    unseen = [(toks[i - 1], toks[i]) for i in range(1, len(toks))
              if _bi_check.ngram[(toks[i - 1], toks[i])] == 0]
    oov = [t for t in toks if t not in _bi_check.vocab and t != BOS]
    print(f"  {s}")
    print(f"     미관측 bigram {len(unseen)}개 {unseen} / 미등록어 {oov}")

print()
print(f"{'모델':<22}{'학습 PPL':>12}{'테스트 PPL':>14}")
print("-" * 48)
for n in [1, 2, 3]:
    for k in [0.0, 1.0]:
        lm = NgramLM(n=n, k=k).fit(train)
        name = f"{n}-gram " + ("MLE" if k == 0 else f"add-{k:g}")
        ppl_tr = lm.perplexity(train)
        ppl_te = lm.perplexity(test)
        f = lambda v: "inf" if v == float("inf") else f"{v:.2f}"
        print(f"{name:<22}{f(ppl_tr):>12}{f(ppl_te):>14}")

#### 측정 결과 해석

위 셀을 실행하면 다음 표가 그대로 재현된다. (코퍼스와 테스트 문장이 고정되어 있으므로 항상 같은 값이 나온다)

| 모델 | 학습 PPL | 테스트 PPL |
|---|---:|---:|
| 1-gram MLE | 15.63 | 18.89 |
| 1-gram add-1 | 15.77 | 18.30 |
| 2-gram MLE | 2.27 | **inf** |
| 2-gram add-1 | 7.29 | 10.75 |
| 3-gram MLE | 1.65 | **inf** |
| 3-gram add-1 | 8.05 | 13.90 |

1. **모든 모델에서 테스트 PPL > 학습 PPL 이다.** (예: 2-gram add-1 은 7.29 → 10.75) 모델은 자기가 본 문장을 가장 잘 예측한다. 두 값의 **차이가 곧 일반화 격차**이며, 언어 모델의 성능은 반드시 학습에 쓰지 않은 문장으로 재야 한다.
2. **MLE 는 미관측 n-gram 이 하나만 있어도 테스트 PPL 이 `inf` 가 된다.** 위 테스트 문장에는 미관측 bigram 이 각각 3개·3개·1개 들어 있어 2-gram MLE·3-gram MLE 모두 `inf` 가 되었다. 확률 0 하나가 문장 전체 확률을 0(로그로는 `-inf`)으로 만들기 때문이다.
3. **스무딩은 학습 PPL 을 오히려 나쁘게 만든다.** 2-gram 의 학습 PPL 은 2.27 → 7.29 로 3배 넘게 올라갔다. 관측된 조합이 갖고 있던 확률 질량을 한 번도 보지 못한 조합에게 나눠 주기 때문이다. 그 대가로 테스트 PPL 이 `inf` → 10.75 로 **유한한 값**이 되었다. 학습 성능을 조금 내주고 일반화를 사는 거래다.
4. **n 을 키운다고 계속 좋아지지 않는다.** add-1 기준 테스트 PPL 은 1-gram 18.30 → 2-gram 10.75 로 좋아지지만, 3-gram 에서는 13.90 으로 다시 나빠진다. 문맥이 길어질수록 가능한 n-gram 종류가 폭증해(희소성) 학습 코퍼스가 이를 감당하지 못하기 때문이다. 이 코퍼스에서 최적 차수는 **2-gram** 이다.
5. **학습 PPL 만 보면 정반대의 결론이 나온다.** 학습 PPL 이 가장 낮은 모델은 3-gram MLE(1.65) 이지만 이 모델의 테스트 PPL 은 `inf` 다. 학습 데이터를 통째로 외운 전형적인 **과적합**이며, 학습 PPL 을 모델 선택 기준으로 쓰면 안 되는 이유를 그대로 보여 준다.

---
# 실습 3. 정보 추출 (Information Extraction)

강의에서 다룬 내용
- 정보 추출 : 비정형 텍스트 -> `(개체1, 관계, 개체2)` 관계형 튜플
- 파이프라인 : 문장 분리 -> 토큰화 -> **NER** -> 상호참조 -> **관계 추출**
- 접근법 : 규칙 기반(Hearst 패턴 등) / 기계학습 기반 / 원격 지도 / Open IE

실습 1에서 만든 NER 결과 위에 **규칙 기반 관계 추출기**를 얹어 파이프라인을 완성한다.

### 3-1. 개체 추출 (파이프라인 1단계)

실습 1의 `rule_based_ner` 를 그대로 재사용하여 문장에서 개체를 뽑는다.

In [ ]:
# 정보 추출 파이프라인의 1단계 : 문서에서 개체를 뽑아 (표층형, 유형, 위치) 형태로 정리한다.

# 소속 · 본사위치 · 출생지 등 서로 다른 관계를 담도록 만든 예제 문서. 2단계(관계 추출)에서도 그대로 쓴다.
DOCS = [
    "홍길동은 2020년부터 한국대학교 컴퓨터공학과 교수로 재직 중이다.",
    "삼성전자는 수원에 본사를 두고 있다.",
    "이순신 장군은 1545년 서울에서 태어났다.",
    "국립국어원은 서울에 위치한 기관이다.",
    "춘향은 남원에서 태어났다.",
]

def extract_entities(sentence):
    """문장 -> [(표층형, 유형, 시작index, 끝index)]"""
    toks = tokenize_ko(sentence)
    tags = rule_based_ner(toks)   # 실습 1에서 만든 규칙 기반 NER을 그대로 재사용한다
    out = []
    for s, e, t in bio_to_spans(tags):
        # 여러 토큰에 걸친 개체는 다시 하나의 문자열로 합쳐 표층형을 만든다.
        surface = " ".join(toks[s:e + 1])
        # 토큰 위치 (s, e)도 함께 남긴다. 2단계에서 두 개체 중 어느 쪽이 앞에 오는지 판단할 때 필요하다.
        out.append((surface, t, s, e))
    return toks, out

for d in DOCS:
    toks, ents = extract_entities(d)
    print(d)
    print("   개체 :", [(w, t) for w, t, _, _ in ents])
    print()

### 3-2. 패턴 기반 관계 추출 (파이프라인 2단계)

관계 추출 규칙은 **개체 유형 쌍 + 문장에 나타나는 단서 표현**으로 정의한다.
아래 표는 다음 셀의 `RELATION_RULES` 와 **한 줄씩 정확히 대응**한다.

| 순서 | 개체1 유형 | 개체2 유형 | 단서 표현 (하나라도 문장에 있으면 발동) | 추출 관계 |
|:---:|:---:|:---:|---|---|
| 1 | PS | OG | `교수` · `재직` · `소속` · `근무` | (PS, **소속**, OG) |
| 2 | OG | LC | `본사` | (OG, **본사_위치**, LC) |
| 3 | OG | LC | `위치한` · `소재` | (OG, **위치**, LC) |
| 4 | PS | LC | `태어났` | (PS, **출생지**, LC) |
| 5 | PS | DT | `태어났` | (PS, **출생연도**, DT) |
| 6 | PS | DT | `재직` · `부터` | (PS, **재직_시작**, DT) |

공통 조건 두 가지에 주의한다.

- **개체1이 개체2보다 문장에서 앞에 나올 때만** 튜플을 만든다. (`s1 < s2`)
- 단서 표현은 두 개체 사이가 아니라 **문장 어디에 있어도** 인정한다. 규칙이 느슨한 만큼 오탐(FP)이 생길 수 있으며, 이는 규칙 기반 방식의 전형적인 한계다.
  (3번 규칙에서 `있다` 를 단서로 쓰지 않은 것도 같은 이유다. 너무 일반적인 단어라 아무 문장에나 걸린다)

In [ ]:
def strip_josa(word):
    """한국어 조사를 제거하여 개체의 표층형을 정규화한다."""
    # 주의 : '와/과'는 '컴퓨터공학과'·'사과'처럼 개체명 자체의 마지막 글자일 수 있어 제외한다
    return re.sub(r"(은|는|이|가|을|를|의|에서|에|으로|로|부터|까지)$", "", word)

RELATION_RULES = [
    # (개체1 유형, 개체2 유형, 문장에 포함되어야 할 단서, 관계명)
    ("PS", "OG", ["교수", "재직", "소속", "근무"], "소속"),
    ("OG", "LC", ["본사"],                        "본사_위치"),
    ("OG", "LC", ["위치한", "소재"],               "위치"),   # '있다'는 너무 일반적이라 오탐을 부른다
    ("PS", "LC", ["태어났"],                      "출생지"),
    ("PS", "DT", ["태어났"],                      "출생연도"),
    ("PS", "DT", ["재직", "부터"],                 "재직_시작"),
]

def extract_relations(sentence):
    toks, ents = extract_entities(sentence)
    text = sentence
    tuples = []
    for i, (w1, t1, s1, e1) in enumerate(ents):
        for j, (w2, t2, s2, e2) in enumerate(ents):
            if i == j:
                continue
            for rt1, rt2, cues, rel in RELATION_RULES:
                if t1 == rt1 and t2 == rt2 and any(c in text for c in cues):
                    if s1 < s2:   # 개체1이 개체2보다 앞에 나오는 경우만 인정
                        tuples.append((strip_josa(w1), rel, strip_josa(w2)))
    # 중복 제거 (순서 유지)
    seen, out = set(), []
    for t in tuples:
        if t not in seen:
            seen.add(t); out.append(t)
    return out


print("=== 관계 추출 결과 ===")
all_tuples = []
for d in DOCS:
    rels = extract_relations(d)
    all_tuples += rels
    print("\n문장 :", d)
    for a, r, b in rels:
        print(f"   ({a}, {r}, {b})")

### 3-3. 추출한 튜플을 구조화된 데이터로 저장

정보 추출의 최종 목표는 **데이터베이스 / 지식 그래프**를 만드는 것이다.  
여기서는 간단히 표 형태로 정리하고, 개체별로 어떤 관계를 갖는지 확인한다.

In [ ]:
# 추출한 (개체1, 관계, 개체2) 튜플을 표와 지식 그래프 형태로 정리한다. 정보 추출의 최종 산출물에 해당한다.
print(f"{'개체1':<20}{'관계':<14}{'개체2':<20}")
print("-" * 54)
for a, r, b in all_tuples:
    print(f"{a:<20}{r:<14}{b:<20}")

print("\n=== 지식 그래프 형태로 정리 ===")
# 같은 주어(head)를 갖는 관계들을 한데 모으면 인접 리스트 형태의 지식 그래프가 된다.
kg = defaultdict(list)
for a, r, b in all_tuples:
    kg[a].append((r, b))
for head, edges in kg.items():
    print(f"{head}")
    for r, b in edges:
        # head --[관계]--> tail 이 지식 그래프의 간선 하나에 해당한다.
        print(f"    --[{r}]--> {b}")

### 3-4. Hearst 패턴을 이용한 상하위(is-a) 관계 추출

관계 집합을 미리 정하지 않고 텍스트 표현 자체를 관계로 쓰는 방식(Open IE)의 가장 고전적인 사례이다.

- 영어 : `NP such as NP, NP, and NP`
- 한국어 : `A, 예를 들어 B, C, D` / `B, C 등의 A`

In [ ]:
# Hearst 패턴으로 상하위(is-a) 관계를 추출한다.
# 관계 종류를 미리 정해 두지 않고 텍스트 표현 자체를 단서로 삼는 Open IE 방식의 고전적인 예이다.

# 영어 : "<상위어> such as <하위어>, <하위어> and <하위어>"
HEARST_EN  = re.compile(r"([a-zA-Z ]+?) such as ((?:[a-zA-Z]+)(?:, [a-zA-Z]+)*(?:,? and [a-zA-Z]+)?)")
# 한국어 1 : "<상위어>, 예를 들어 <하위어 목록>"
HEARST_KO1 = re.compile(r"([가-힣]+)\s*,?\s*예를 들어\s*([가-힣]+(?:\s*,\s*[가-힣]+)*)")
# 한국어 2 : "<하위어 목록> 등의 <상위어>"  — 목록이 앞에 오므로 그룹 순서가 반대이다.
HEARST_KO2 = re.compile(r"([가-힣]+(?:\s*,\s*[가-힣]+)*)\s*등의\s*([가-힣]+)")

def _clean_list(s):
    """'사과, 배, 감은' -> ['사과','배','감'] : 조사는 목록의 마지막 항목에만 붙으므로 마지막만 정규화한다."""
    items = [x.strip() for x in s.split(",") if x.strip()]
    if items:
        items[-1] = strip_josa(items[-1]) or items[-1]
    return items

# 위 세 패턴이 각각 하나씩 걸리도록 만든 예문
SENTS = [
    "Fruits such as apples, pears and grapes are rich in vitamins.",
    "과일, 예를 들어 사과, 배, 감은 비타민이 풍부하다.",
    "사과, 배, 감 등의 과일은 비타민이 풍부하다.",
]

def hearst(sent):
    """Hearst 패턴으로 (하위어, is-a, 상위어) 튜플을 추출한다."""
    out = []
    # 영어 패턴 : group(1)이 상위어, group(2)가 쉼표와 and 로 이어진 하위어 목록이다.
    m = HEARST_EN.search(sent)
    if m:
        hyper = m.group(1).strip()
        # 목록을 쉼표와 " and " 기준으로 잘라 하위어를 하나씩 분리한다.
        for h in re.split(r",| and ", m.group(2)):
            h = h.strip()
            if h:
                out.append((h, "is-a", hyper))
    # 한국어는 상위어에도 조사가 붙으므로("과일은") 조사를 떼어 내 표층형을 통일한다.
    m = HEARST_KO1.search(sent)
    if m:
        hyper = strip_josa(m.group(1).strip())
        for h in _clean_list(m.group(2)):
            out.append((h, "is-a", hyper))
    m = HEARST_KO2.search(sent)
    if m:
        hyper = strip_josa(m.group(2).strip())
        for h in _clean_list(m.group(1)):
            out.append((h, "is-a", hyper))
    return out

for s in SENTS:
    print(s)
    for t in hearst(s):
        print("   ", t)
    print()

---
# 📌 제출 과제 (총 100점)

아래 세 문항을 모두 수행한 뒤, **모든 셀을 위에서부터 실행하여 출력이 보이는 상태로** 저장해 제출한다.

- 파일명 : `7주차_과제_학번_이름.ipynb`
- 각 과제 아래의 **채점 셀** 을 실행해 `통과!` 가 출력되는지 반드시 확인한다.
- 평가 기준 : 코드의 정확한 동작(60%) · 결과에 대한 해석과 설명(40%)
- **코드만 있고 설명이 없으면 감점**된다. 각 문항의 '설명' 마크다운 셀을 반드시 채울 것.

## 과제 1. BIOES 태깅 변환기 구현 (30점) — 7.1절

강의에서 살펴본 **BIOES(=BILOU)** 표현으로 변환하는 함수를 작성하시오.

| 태그 | 의미 |
|---|---|
| `B-` | 두 토큰 이상 개체의 **시작** |
| `I-` | 두 토큰 이상 개체의 **중간** |
| `E-` | 두 토큰 이상 개체의 **끝** |
| `S-` | **한 토큰**으로 이루어진 개체 |
| `O` | 개체 아님 |

**요구 사항**

1. `bio_to_bioes(tags)` : BIO 태그열 → BIOES 태그열
   예) `["B-OG", "I-OG"]` → `["B-OG", "E-OG"]`,  `["B-PS"]` → `["S-PS"]`
2. `bioes_to_bio(tags)` : BIOES 태그열 → BIO 태그열 (왕복 변환이 원래 태그열과 정확히 일치해야 한다)
3. 실습 1의 `bio_to_spans` 를 재사용하면 훨씬 간단해진다.

In [ ]:
# ── 과제 1 ────────────────────────────────────────────────────────
def bio_to_bioes(tags):
    """BIO 태그 리스트를 BIOES 태그 리스트로 변환한다."""
    # TODO 1 : bio_to_spans(tags) 로 (start, end, type) 목록을 먼저 구한다.
    # TODO 2 : start == end 인 한 토큰짜리 개체는 "S-유형" 으로 표시한다.
    # TODO 3 : 두 토큰 이상 개체는 시작을 "B-", 끝을 "E-", 그 사이를 "I-" 로 채운다.
    # TODO 4 : 개체가 아닌 자리는 "O" 로 남긴다. (입력과 길이가 같아야 한다)
    raise NotImplementedError('과제 1-1 : BIO 태그열을 BIOES(S-/B-/I-/E-/O) 태그열로 바꾸는 bio_to_bioes 를 구현하시오.')


def bioes_to_bio(tags):
    """BIOES 태그 리스트를 다시 BIO 태그 리스트로 되돌린다."""
    # TODO 5 : "S-" 는 "B-" 로, "E-" 는 "I-" 로 되돌리고 "B-"/"I-"/"O" 는 그대로 둔다.
    raise NotImplementedError('과제 1-2 : BIOES 태그열을 BIO 태그열로 되돌리는 bioes_to_bio 를 구현하시오.')


# 실습 1의 GOLD_DATA 에 적용해 보는 데모
try:
    for toks, gold in GOLD_DATA[:3]:
        print(" ".join(toks))
        print("   BIO   :", gold)
        print("   BIOES :", bio_to_bioes(gold))
        print()
except NotImplementedError as e:
    print('▶ 미구현 :', e)

In [ ]:
# [과제 1] 채점 — 기대 조건이 하드코딩되어 있다.
try:
    _cases = [
        ["B-PS", "O", "B-DT", "I-DT", "B-LC"],
        ["B-OG", "I-OG", "I-OG", "O"],
        ["O", "O"],
        ["B-PS", "B-PS"],
    ]
    _want = [
        ["S-PS", "O", "B-DT", "E-DT", "S-LC"],
        ["B-OG", "I-OG", "E-OG", "O"],
        ["O", "O"],
        ["S-PS", "S-PS"],
    ]
    for _src, _exp in zip(_cases, _want):
        _got = bio_to_bioes(_src)
        assert isinstance(_got, list), 'bio_to_bioes 는 리스트를 반환해야 합니다.'
        assert len(_got) == len(_src), f'입력과 같은 길이여야 합니다. 입력 {_src} → 출력 길이 {len(_got)}'
        assert all(isinstance(x, str) for x in _got), '태그는 문자열이어야 합니다.'
        assert _got != _src or _src == _exp, f'입력을 그대로 돌려주고 있습니다. (입력 {_src})'
        assert _got == _exp, f'BIOES 변환 결과가 틀렸습니다. 입력 {_src} → 출력 {_got} 를 다시 확인하시오.'

    # 네 토큰짜리 개체 : B- ... I- ... E- 구조가 되는지 (S- 남발/전부 I- 채우기 차단)
    _long = ["B-OG", "I-OG", "I-OG", "I-OG"]
    _g = bio_to_bioes(_long)
    assert _g[0] == "B-OG" and _g[-1] == "E-OG" and set(_g[1:-1]) == {"I-OG"}, \
        f'긴 개체는 시작 B-, 끝 E-, 중간 I- 여야 합니다. (출력 {_g})'

    # 왕복 변환 검사 — 반드시 try 안에서 수행한다 (미구현일 때 ▶ 미구현 만 출력되도록)
    for _src in _cases + [_long]:
        assert bioes_to_bio(bio_to_bioes(_src)) == _src, \
            f'왕복 변환(BIO→BIOES→BIO)이 원래 태그열과 달라졌습니다. (입력 {_src})'
    for _toks, _gold in GOLD_DATA:
        assert bioes_to_bio(bio_to_bioes(_gold)) == _gold, \
            f'GOLD_DATA 문장에서 왕복 변환이 실패했습니다. ({" ".join(_toks)})'

    print('통과!  테스트 5종 +  GOLD_DATA', len(GOLD_DATA), '문장의 왕복 변환까지 확인 완료')
except NotImplementedError as e:
    print('▶ 미구현 :', e)

**과제 1 설명** (이 셀에 서술하시오)

- BIO 대신 BIOES 를 쓰면 무엇이 좋아지는가? (개체의 끝을 명시하는 것이 왜 도움이 되는가)
- 태그 종류 수는 개체 유형이 N개일 때 BIO 는 `2N+1` 개다. BIOES 는 몇 개인가? 그 대가는 무엇인가?

## 과제 2. add-k 스무딩과 보간법 비교 (40점) — 7.2절

**(2-1)** `k` 값을 바꿔 가며 학습·테스트 퍼플렉서티가 어떻게 변하는지 표로 출력하시오.
`k` 가 너무 작으면 미관측 조합에 확률이 거의 배정되지 않고, 너무 크면 관측된 조합의 확률까지 뭉개진다.

**(2-2)** 강의에서 배운 **선형 보간법(interpolation)** 을 구현하시오.

`P_interp(w | w1, w2) = λ3 · P3(w | w1, w2) + λ2 · P2(w | w2) + λ1 · P1(w)`,  단 `λ1 + λ2 + λ3 = 1`

> 두 문항 모두 실습 2에서 만든 `NgramLM` 과 `BIG_CORPUS`, `test` 를 그대로 사용한다.

In [ ]:
# ── 과제 2-1 ──────────────────────────────────────────────────────
def sweep_add_k(train_sents, test_sents, ks=(0.001, 0.01, 0.1, 0.5, 1.0)):
    """add-k 의 k 를 바꿔 가며 2-gram 모델의 학습/테스트 퍼플렉서티를 표로 출력하고
       [(k, 학습 PPL, 테스트 PPL), ...] 을 ks 와 같은 순서로 반환한다."""
    # TODO 1 : ks 의 각 k 마다 NgramLM(n=2, k=k) 을 train_sents 로 새로 학습한다.
    #          (모델을 한 번만 만들어 재사용하면 k 를 바꾼 효과가 나타나지 않는다)
    # TODO 2 : lm.perplexity(train_sents) 와 lm.perplexity(test_sents) 를 각각 계산한다.
    # TODO 3 : k / 학습 PPL / 테스트 PPL 을 한 줄씩 표 형태로 출력한다.
    # TODO 4 : [(k, 학습 PPL, 테스트 PPL), ...] 리스트를 반환한다.
    raise NotImplementedError('과제 2-1 : k 를 바꿔 가며 2-gram 모델의 학습·테스트 퍼플렉서티를 표로 출력하고 [(k, 학습 PPL, 테스트 PPL)] 을 반환하시오.')


try:
    k_rows = sweep_add_k(BIG_CORPUS, test)
    best_k = min(k_rows, key=lambda r: r[2])[0]
    print(f"\n테스트 퍼플렉서티가 가장 낮은 k = {best_k}")
except NotImplementedError as e:
    k_rows, best_k = [], None
    print('▶ 미구현 :', e)

In [ ]:
# [과제 2-1] 채점 — 기대 조건이 하드코딩되어 있다.
try:
    _ks = (0.001, 0.01, 0.1, 0.5, 1.0)
    _rows = sweep_add_k(BIG_CORPUS, test, _ks)
    assert isinstance(_rows, (list, tuple)) and len(_rows) == len(_ks), \
        f'{len(_ks)}개의 행 [(k, 학습 PPL, 테스트 PPL)] 을 반환해야 합니다.'
    assert all(hasattr(r, '__len__') and len(r) == 3 for r in _rows), \
        '각 행은 (k, 학습 PPL, 테스트 PPL) 세 값이어야 합니다.'
    assert [float(r[0]) for r in _rows] == list(_ks), f'k 값과 순서는 {list(_ks)} 여야 합니다.'

    _tr = [float(r[1]) for r in _rows]
    _te = [float(r[2]) for r in _rows]
    assert all(math.isfinite(v) and v > 1 for v in _tr + _te), \
        'k > 0 이면 확률이 0 이 될 수 없으므로 PPL 은 모두 유한한 값(>1)이어야 합니다.'
    assert len(set(round(v, 6) for v in _te)) >= 4, \
        'k 를 바꿨는데 테스트 PPL 이 거의 변하지 않습니다. k 마다 모델을 다시 학습했는지 확인하시오.'
    assert len(set(round(v, 6) for v in _tr)) >= 4, \
        'k 를 바꿨는데 학습 PPL 이 거의 변하지 않습니다. k 가 확률 계산에 실제로 반영되는지 확인하시오.'
    assert _tr == sorted(_tr), \
        'k 가 커질수록 학습 PPL 은 단조 증가해야 합니다. (확률 질량이 미관측 조합으로 분산되므로)'
    _best = min(range(len(_te)), key=lambda i: _te[i])
    assert 0 < _best < len(_te) - 1, \
        ('테스트 PPL 은 너무 작은 k(과소 평활)와 너무 큰 k(과대 평활) 사이에서 최소가 되어야 합니다. '
         f'현재는 양 끝인 k={_ks[_best]} 에서 최소입니다. 계산을 다시 확인하시오.')

    print(f'통과!  테스트 PPL 최소 지점 k = {_ks[_best]} '
          f'(학습 PPL {_tr[_best]:.2f} / 테스트 PPL {_te[_best]:.2f})')
    print(f'  참고) k=1.0(라플라스) 의 테스트 PPL 은 {_te[-1]:.2f} 이다. 왜 더 나쁜지 설명 셀에 서술한다.')
except NotImplementedError as e:
    print('▶ 미구현 :', e)

In [ ]:
# ── 과제 2-2 ──────────────────────────────────────────────────────
class InterpolatedLM:
    """1-gram, 2-gram, 3-gram 확률을 선형 결합하는 보간 언어 모델."""

    def __init__(self, lambdas=(0.1, 0.3, 0.6)):
        assert abs(sum(lambdas) - 1.0) < 1e-9, "가중치의 합은 1이어야 함"
        self.l1, self.l2, self.l3 = lambdas

    def fit(self, sentences):
        self.uni = NgramLM(n=1, k=0.01).fit(sentences)
        self.bi  = NgramLM(n=2, k=0.01).fit(sentences)
        self.tri = NgramLM(n=3, k=0.01).fit(sentences)
        return self

    def prob(self, word, context):
        """context : 앞의 두 단어 튜플 (w_{i-2}, w_{i-1})"""
        # TODO 1 : 3-gram 확률 P3(word | w_{i-2}, w_{i-1}) 을 self.tri.prob 으로 구한다.
        # TODO 2 : 2-gram 확률 P2(word | w_{i-1}) 과 1-gram 확률 P1(word) 을 구한다.
        #          (self.bi 에는 직전 한 단어만, self.uni 에는 빈 문맥 () 을 넘긴다)
        # TODO 3 : self.l3 * P3 + self.l2 * P2 + self.l1 * P1 을 반환한다.
        raise NotImplementedError('과제 2-2 : 3-gram·2-gram·1-gram 확률을 λ 로 가중 합하는 InterpolatedLM.prob 을 구현하시오.')

    def perplexity(self, sentences):
        logsum, ntok = 0.0, 0
        for s in sentences:
            toks = [BOS, BOS] + tokenize_en(s.lower()) + [EOS]
            for i in range(2, len(toks)):
                p = self.prob(toks[i], (toks[i-2], toks[i-1]))
                if p <= 0:
                    return float("inf")
                logsum += math.log(p); ntok += 1
        return math.exp(-logsum / ntok)


try:
    interp = InterpolatedLM((0.1, 0.3, 0.6)).fit(BIG_CORPUS)
    print(f"P(boy | <s> the)        = {interp.prob('boy', ('<s>', 'the')):.5f}")
    print(f"보간 모델 테스트 PPL     = {interp.perplexity(test):.2f}")
    base_tri = NgramLM(n=3, k=1.0).fit(BIG_CORPUS)
    print(f"3-gram add-1 테스트 PPL = {base_tri.perplexity(test):.2f}")
except NotImplementedError as e:
    interp = None
    print('▶ 미구현 :', e)

In [ ]:
# [과제 2-2] 채점 — 기대 조건이 하드코딩되어 있다.
# 주의 : "테스트 PPL 이 낮다" 는 것만으로는 정답을 가릴 수 없다.
#        예컨대 항상 0.5 를 돌려주는 가짜 구현은 PPL 이 2.00 으로 '더 좋아' 보인다.
#        그래서 아래에서는 값이 아니라 '확률 분포인가' 를 직접 검사한다.
try:
    _m = InterpolatedLM((0.1, 0.3, 0.6)).fit(BIG_CORPUS)
    assert abs((_m.l1 + _m.l2 + _m.l3) - 1.0) < 1e-9, '보간 가중치 λ1+λ2+λ3 은 1이어야 합니다.'

    _vocab = sorted(_m.tri.vocab)
    _ctx_a, _ctx_b = ("<s>", "the"), ("is", "looking")

    _p = float(_m.prob("boy", _ctx_a))
    assert 0 < _p < 1, f'확률은 0과 1 사이여야 합니다. (현재 {_p})'

    # (1) 한 문맥에서 어휘 전체에 대한 확률의 합이 1이어야 한다.
    #     상수를 돌려주는 가짜 구현은 여기서 반드시 걸린다.
    for _ctx in (_ctx_a, _ctx_b):
        _total = sum(float(_m.prob(w, _ctx)) for w in _vocab)
        assert abs(_total - 1.0) < 1e-6, \
            (f'문맥 {_ctx} 에서 어휘 {len(_vocab)}개에 대한 확률의 합이 {_total:.4f} 입니다. '
             '확률 분포라면 합이 1이어야 합니다. (상수를 돌려주거나 정규화가 깨진 구현)')

    # (2) 문맥이 바뀌면 분포도 바뀌어야 한다. (문맥을 무시하는 구현 차단)
    _da = [float(_m.prob(w, _ctx_a)) for w in _vocab]
    _db = [float(_m.prob(w, _ctx_b)) for w in _vocab]
    assert max(abs(x - y) for x, y in zip(_da, _db)) > 1e-3, \
        f'문맥 {_ctx_a} 와 {_ctx_b} 의 확률 분포가 동일합니다. 문맥을 사용하지 않는 구현입니다.'

    # (3) λ 가 실제로 반영되어야 한다.
    _m2 = InterpolatedLM((0.6, 0.3, 0.1)).fit(BIG_CORPUS)
    assert abs(float(_m2.prob("boy", _ctx_a)) - _p) > 1e-6, \
        'λ 를 바꿔도 확률이 그대로입니다. self.l1/self.l2/self.l3 를 사용하지 않았습니다.'

    # (4) 3-gram 문맥 정보가 살아 있어야 한다.
    assert float(_m.prob("boy", _ctx_a)) > float(_m.prob("park", _ctx_a)), \
        '"<s> the" 다음에 실제로 관측된 단어의 확률이 더 높아야 합니다. 3-gram 항이 빠졌습니다.'

    _ppl = _m.perplexity(test)
    assert math.isfinite(_ppl) and _ppl > 1, '보간 모델의 테스트 PPL 은 유한한 값이어야 합니다.'
    _base = NgramLM(n=3, k=1.0).fit(BIG_CORPUS).perplexity(test)
    print(f'통과!  보간 모델 테스트 PPL = {_ppl:.2f} / 3-gram add-1 테스트 PPL = {_base:.2f}')
    print('  (두 값의 차이가 무엇을 뜻하는지는 아래 설명 셀에 직접 서술한다)')
except NotImplementedError as e:
    print('▶ 미구현 :', e)

**과제 2 설명** (이 셀에 서술하시오)

- (2-1) `k` 를 키우면 학습 PPL 과 테스트 PPL 이 각각 어떻게 움직였는가? 테스트 PPL 이 최소가 되는 `k` 가 양 끝이 아니라 중간에 있는 이유는 무엇인가?
- (2-1) 라플라스(`k=1`)가 실제 연구에서 잘 쓰이지 않는 이유를 위 표의 숫자를 근거로 설명하시오.
- (2-2) 보간 모델과 3-gram add-1 모델의 테스트 PPL 을 비교하고, 차이가 생긴 이유를 서술하시오.
- (2-2) `λ3` 를 키우면(3-gram 을 더 신뢰하면) 결과가 어떻게 달라질지 예상하고, 직접 실험해 확인하시오.

## 과제 3. 한국어 문서에서 관계 튜플 추출 (30점) — 7.3절

아래 `HOMEWORK_DOCS` 에서 **최소 6개 이상의 관계 튜플**을 추출하시오.

**요구 사항**

1. `GAZETTEER` 에 문서에 등장하는 개체명을 유형별로 추가한다.
2. `RELATION_RULES` 에 새 규칙을 추가한다. (예: `설립자`, `설립연도`, `창제연도` …)
   - 힌트 : 기존 출생지 규칙의 단서는 `태어났` 이라 `태어난` 에는 걸리지 않는다.
3. `extract_relations` 를 실행하여 문서별 결과를 출력한다.
4. 잘못 추출된 튜플(FP)이나 놓친 관계(FN)가 있다면 무엇이고 왜 그런지 설명 셀에 서술한다.

In [ ]:
# ── 과제 3 ────────────────────────────────────────────────────────
HOMEWORK_DOCS = [
    "세종대왕은 1443년에 훈민정음을 창제하였다.",
    "네이버는 1999년 이해진이 설립한 기업이다.",
    "카카오는 제주에 본사를 두고 있다.",
    "김소월은 1902년 평안북도에서 태어난 시인이다.",
    "한국대학교는 서울에 위치한 대학교이다.",
    "이순신은 1592년 한산도에서 큰 승리를 거두었다.",
]


def extend_rules():
    """HOMEWORK_DOCS 를 처리할 수 있도록 GAZETTEER 와 RELATION_RULES 를 확장한다."""
    # TODO 1 : GAZETTEER["PS"], ["OG"], ["LC"] 에 위 문서에 등장하는 개체명을 추가한다.
    #          (이미 들어 있는 항목을 다시 넣지 않도록 검사한 뒤 append 할 것)
    # TODO 2 : RELATION_RULES 에 (개체1 유형, 개체2 유형, 단서 목록, 관계명) 형태의 새 규칙을 추가한다.
    #          서로 다른 관계명이 최소 3종류는 나오도록 규칙을 설계한다.
    raise NotImplementedError('과제 3-1 : GAZETTEER 와 RELATION_RULES 를 확장해 HOMEWORK_DOCS 의 개체·관계를 잡을 수 있게 하시오.')


def extract_all(docs):
    """docs 의 모든 문장에서 (개체1, 관계, 개체2) 튜플을 뽑아 하나의 리스트로 모아 반환한다."""
    # TODO 3 : 문장마다 extract_relations(d) 를 호출하고 결과를 이어 붙여 반환한다.
    raise NotImplementedError('과제 3-2 : 각 문서에 extract_relations 를 적용해 관계 튜플 전체를 리스트로 반환하시오.')


try:
    extend_rules()
    hw_tuples = extract_all(HOMEWORK_DOCS)
    for d in HOMEWORK_DOCS:
        print(d)
        for t in extract_relations(d):
            print("   ", t)
        print()
    print("총 추출 튜플 수 :", len(hw_tuples))
except NotImplementedError as e:
    hw_tuples = []
    print('▶ 미구현 :', e)

In [ ]:
# [과제 3] 채점 — 기대 조건이 하드코딩되어 있다.
try:
    extend_rules()                       # 규칙이 반영된 상태를 보장한다
    _res = extract_all(HOMEWORK_DOCS)
    assert isinstance(_res, list), 'extract_all 은 리스트를 반환해야 합니다.'
    assert all(isinstance(t, tuple) and len(t) == 3 for t in _res), \
        '각 원소는 (개체1, 관계, 개체2) 3-튜플이어야 합니다.'
    assert all(all(isinstance(x, str) and x.strip() for x in t) for t in _res), \
        '튜플의 세 값은 모두 비어 있지 않은 문자열이어야 합니다.'
    assert len(_res) >= 6, f'관계 튜플이 6개 이상이어야 합니다. (현재 {len(_res)}개)'
    assert len(set(_res)) == len(_res), '같은 튜플이 중복되어 있습니다.'

    _rels = {r for _, r, _ in _res}
    assert len(_rels) >= 3, f'서로 다른 관계가 3종류 이상이어야 합니다. (현재 {sorted(_rels)})'
    _rule_names = {r[3] for r in RELATION_RULES}
    assert _rels <= _rule_names, \
        f'RELATION_RULES 에 없는 관계명이 있습니다: {sorted(_rels - _rule_names)}. 규칙을 추가해 추출하시오.'
    assert len({a for a, _, _ in _res}) >= 4, '서로 다른 주어(개체1)가 4개 이상 나와야 합니다.'

    # 손으로 적은 튜플이 아니라 실제 파이프라인이 만들어 낸 결과여야 한다.
    _pipe = set()
    for _d in HOMEWORK_DOCS:
        _pipe |= set(extract_relations(_d))
    assert set(_res) <= _pipe, \
        'extract_relations 가 만들어 내지 못하는 튜플이 섞여 있습니다. 결과를 직접 적어 넣으면 안 됩니다.'

    # 조사가 붙은 채로 남아 있으면 지식 그래프의 같은 개체가 서로 다른 노드가 된다.
    _josa = ('은', '는', '이', '가', '을', '를', '의', '에서', '에', '으로', '부터')
    _bad = [t for t in _res if t[0].endswith(_josa) or t[2].endswith(_josa)]
    assert not _bad, f'개체 표층형에 조사가 남아 있습니다: {_bad[:3]} (strip_josa 를 적용하시오)'

    _text = " ".join(HOMEWORK_DOCS)
    assert all(a in _text and b in _text for a, _, b in _res), \
        '문서에 등장하지 않는 문자열이 개체로 들어 있습니다.'

    print(f'통과!  관계 튜플 {len(_res)}개, 관계 종류 {len(_rels)}개 : {sorted(_rels)}')
    print(f'  예시 : {_res[:3]}')
except NotImplementedError as e:
    print('▶ 미구현 :', e)

**과제 3 설명** (이 셀에 서술하시오)

- 어떤 개체명·규칙을 추가했는가? 규칙 하나를 추가할 때마다 어떤 문장이 새로 잡혔는지 적으시오.
- 잘못 추출된 튜플(FP)이 있었다면 무엇이며, 규칙의 어떤 점 때문에 생겼는가? (단서 표현이 문장 어디에 있어도 인정된다는 점을 떠올릴 것)
- 놓친 관계(FN)는 무엇인가? 사전에 없는 개체명·어미 변화(`태어났/태어난`) 때문에 놓친 사례를 찾아 적으시오.
- 규칙 기반 관계 추출의 정밀도가 높고 재현율이 낮은 이유를 위 결과를 근거로 설명하시오.

---
## 마무리 : 스스로 점검하기

- [ ] BIO 태깅에서 태그 개수가 `2N+1` 개가 되는 이유를 설명할 수 있는가?
- [ ] NER 평가에 정확도(accuracy) 대신 F1을 쓰는 이유를 설명할 수 있는가?
- [ ] strict 평가와 relaxed 평가의 결과 차이를 설명할 수 있는가?
- [ ] N을 키울 때 학습 PPL과 테스트 PPL이 서로 다르게 움직이는 이유를 설명할 수 있는가?
- [ ] 라플라스 스무딩이 실제로는 잘 쓰이지 않는 이유를 설명할 수 있는가?
- [ ] 규칙 기반 관계 추출의 정밀도가 높고 재현율이 낮은 이유를 설명할 수 있는가?

수고하셨습니다.